In [ ]:
from langgraph.graph import StateGraph, MessagesState,START, END,add_messages
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage,SystemMessage
from typing import TypedDict,Annotated,List,Sequence
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import ServerlessSpec,Pinecone
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from langgraph.prebuilt import ToolNode,tools_condition
from langfuse import get_client
from langfuse.langchain import CallbackHandler

d:\ProdRAG\prodRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
pc=Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "practice-rag" 

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [4]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.5
)

In [5]:
class RAGState(TypedDict):
    messages:Annotated[List[BaseMessage],add_messages]

In [6]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [7]:
rag_prompt = PromptTemplate.from_template(
   template= """You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [8]:
file_path = r"D:\ProdRAG\prodRAG\Blockchain_Course_Proposal.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [10]:
embedder = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview",output_dimensionality=384)


In [11]:
vectorstore = PineconeVectorStore.from_documents(
    texts,
    embedder,
    index_name=index_name
)

In [12]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
@tool
def rag_tool(query: str) -> str:
    '''A tool that retrieves relevant information from the PDF based on a user query.
    Use this tool when the user asks factual/conceptual questions that might be answered from the stored documents.'''
    
    # Retrieve similar chunks from the vectorstore
    docs = retriever.invoke(query)
    context = format_docs(docs)
    
    return context

In [14]:
tools=[rag_tool]
llm_with_tools=llm.bind_tools(tools)

In [ ]:
def chat_rag(state:RAGState):
    # Create a system message that instructs the LLM to use tool results
    system_prompt = """You are a helpful assistant. When you use the rag_tool, carefully read the context it returns and base your answer ONLY on that context.
If the context doesn't contain the answer, say 'I don't know based on the provided context.' Do NOT use your general knowledge if the tool provides information."""
    
    # Prepare messages with system context
    messages_with_system = [SystemMessage(content=system_prompt)] + state["messages"]
    
    response = llm_with_tools.invoke(messages_with_system)
    return {"messages": [response]}

In [16]:
tool_node=ToolNode(tools)

In [17]:
workflow=StateGraph(RAGState)
workflow.add_node("chat_rag",chat_rag)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "chat_rag")
workflow.add_conditional_edges(
    "chat_rag",
    tools_condition,
    {"tools": "tools",END: END}

)
workflow.add_edge("tools", "chat_rag")      

app=workflow.compile()

In [18]:
langfuse = get_client()
langfuse_handler = CallbackHandler()

In [19]:
result=app.invoke(
    {"messages": [HumanMessage(content=("What is module1?"))]},
    # config={"callbacks": [langfuse_handler]}
)

In [20]:
print(result)

{'messages': [HumanMessage(content='What is module1?', additional_kwargs={}, response_metadata={}, id='d39859c2-282c-42e3-a671-44b5ac5ae584'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6s3njnhye', 'function': {'arguments': '{"query":"module1"}', 'name': 'rag_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 298, 'total_tokens': 314, 'completion_time': 0.029657441, 'completion_tokens_details': None, 'prompt_time': 0.031051124, 'prompt_tokens_details': None, 'queue_time': 0.049364388, 'total_time': 0.060708565}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4cbb-0be5-7550-8d23-ceb3e386a800-0', tool_calls=[{'name': 'rag_tool', 'args': {'query': 'module1'}, 'id': '6s3njnhye', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 298, 'out

In [21]:
print(result["messages"][-1].content)

It seems like the rag_tool was unable to find any information about module1.
